In [0]:
from pyspark.sql.functions import col, count, sum, when, to_date, explode, sequence, coalesce, current_date, lit, sum, when, countDistinct
from pyspark.sql import Row
from functools import reduce

In [0]:
catalog = "subscription_platform"
silver_schema = "silver"

silver_events = f"{catalog}.{silver_schema}.subscription_events_clean"
silver_subscriptions_history = f"{catalog}.{silver_schema}.subscriptions_history"

In [0]:
history_df = spark.read.table(silver_subscriptions_history)
events_df = spark.read.table(silver_events)

In [0]:
gold_schema = "gold"


### Gold -> Daily Activity Metrics

In [0]:
gold_daily_activity_metrics_table = f"{catalog}.{gold_schema}.daily_activity_metrics"
gold_daily_activity_metrics_path = "s3://subscription-revenue-platform/gold/daily_revenue_metrics/"

In [0]:
new_subs = (
    events_df
    .filter(col("event_type") == "subscription_created")
    .groupBy(col("event_date").alias("metric_date"))
    .agg(count("*").alias("new_subscriptions"))
)

display(new_subs)

metric_date,new_subscriptions
2026-04-06,2
2026-04-09,14
2026-04-07,1
2026-04-10,7
2026-04-08,2


In [0]:
cancellations = (
    events_df
    .filter(col("event_type") == "subscription_cancelled")
    .groupBy(col("event_date").alias("metric_date"))
    .agg(count("*").alias("cancellations"))
)

display(cancellations)

metric_date,cancellations
2026-04-06,2
2026-04-09,8
2026-04-10,4
2026-04-08,2


In [0]:
plan_changes = (
    events_df
    .filter(col("event_type") == "plan_changed")
    .groupBy(col("event_date").alias("metric_date"))
    .agg(count("*").alias("plan_changes"))
)

display(plan_changes)

metric_date,plan_changes
2026-04-07,4
2026-04-09,7
2026-04-10,12


In [0]:
dfs = [new_subs, cancellations, plan_changes]

daily_activity_metrics = reduce(
    lambda left, right: left.join(right, on="metric_date", how="outer"),
    dfs
).fillna(0)

display(daily_activity_metrics)

metric_date,new_subscriptions,cancellations,plan_changes
2026-04-07,1,0,4
2026-04-09,14,8,7
2026-04-10,7,4,12
2026-04-06,2,2,0
2026-04-08,2,2,0


In [0]:
(
daily_activity_metrics.write
    .format("delta")
    .mode("append")
    .saveAsTable(gold_daily_activity_metrics_table)
)

In [0]:
%sql

SELECT *
FROM subscription_platform.gold.daily_activity_metrics
ORDER BY metric_date;

metric_date,new_subscriptions,cancellations,plan_changes
2026-04-06,2,2,0
2026-04-07,1,0,4
2026-04-08,2,2,0
2026-04-09,14,8,7
2026-04-10,7,4,12



### Gold -> Subscription KPIs Snapshot

In [0]:
gold_subscription_kpis_snapshot_table = f"{catalog}.{gold_schema}.subscription_kpis_snapshot"
gold_subscription_kpis_snapshot_path = "s3://subscription-revenue-platform/gold/subscription_kpis_snapshot/"

In [0]:
snapshot_df = (
    history_df
    .agg(
        sum(when((col("is_current") == True) & (col("status") == "active"), col("amount"))).alias("mrr"),
        sum(when((col("is_current") == True) & (col("status") == "active"), 1).otherwise(0)).alias("active_subscriptions"),
        sum(when((col("is_current") == True) & (col("status") == "cancelled"), 1).otherwise(0)).alias("cancelled_subscriptions"),
        countDistinct(col("subscription_id")).alias("total_subscriptions")
    )
    .withColumn("snapshot_date", current_date())
    .select(
        "snapshot_date",
        "mrr",
        col("active_subscriptions").cast("int"),
        col("cancelled_subscriptions").cast("int"),
        col("total_subscriptions").cast("int")
    )
)

display(snapshot_df)

snapshot_date,mrr,active_subscriptions,cancelled_subscriptions,total_subscriptions
2026-04-15,12000.0,8,1,9


In [0]:
snapshot_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(gold_subscription_kpis_snapshot_table)

In [0]:
%sql

SELECT *
FROM subscription_platform.gold.subscription_kpis_snapshot
ORDER BY snapshot_date;

snapshot_date,mrr,active_subscriptions,cancelled_subscriptions,total_subscriptions
2026-04-15,12000.0,8,1,9
